In [1]:
# ============================================================
# PS S6E5 - exp_20260523_049_xgb_shared001v2_features_optuna_search_min
#
# Type:
#   Optuna search only. This notebook does NOT create final OOF/pred/submission.
#
# Base:
#   041: exp_20260520_041_xgb_shared001v2_features_min
#
# Purpose:
#   Keep the 041 XGB shared001v2 feature set fixed and search only XGBoost
#   hyperparameters with Optuna.
#
# Important:
#   - No new feature engineering.
#   - No 039 LOG features.
#   - Normalized_TyreLife direct/proxy use remains forbidden.
#   - Original rows are appended only to each fold train side, same as 041.
#   - Search uses a single model seed to reduce cost.
#   - If search is successful, create 050 refit using best_params_049.json.
#
# Baselines:
#   016 XGB: CV 0.9519855866173828 / Public LB 0.95164
#   041 XGB: CV 0.9520124407043763 / Public LB 0.95173
#
# Success criteria:
#   - best_cv_auc > 0.9520124407043763
#   - strong if best_cv_auc >= 0.95220
# ============================================================


In [2]:
import os
import gc
import json
import random
import warnings
import pickle
from datetime import datetime
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
import optuna

warnings.simplefilter("ignore")
pd.set_option("display.max_columns", 300)

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, log_loss

import cudf
from cuml.preprocessing import TargetEncoder as cuTargetEncoder

from xgboost import XGBClassifier


In [3]:
# ============================================================
# Config
# ============================================================

class CFG:
    EXP_ID = "exp_20260523_049_xgb_shared001v2_features_optuna_search_min"
    COMPETITION = "PS S6E5 Predicting F1 Pit Stops"
    METRIC = "AUC"
    CREATED_AT = "2026-05-23"

    TARGET_CANDIDATES = ["PitNextLap", "PitNextLab"]
    TARGET = "PitNextLap"
    ID_COL = "id"
    DANGER_COL = "Normalized_TyreLife"

    COMP_PATHS = [
        "/kaggle/input/competitions/playground-series-s6e5",
        "/kaggle/input/playground-series-s6e5",
        ".",
    ]

    ORIGINAL_PATHS = [
        "/kaggle/input/datasets/aadigupta1601/f1-strategy-dataset-pit-stop-prediction/f1_strategy_dataset_v4.csv",
        "/kaggle/input/f1-strategy-dataset-pit-stop-prediction/f1_strategy_dataset_v4.csv",
        "./f1_strategy_dataset_v4.csv",
    ]

    OUTDIR = Path(f"/kaggle/working/{EXP_ID}")

    SEED = 42
    N_SPLITS = 5

    N_TE_FOLDS = 5
    TE_SMOOTH = 20

    # 049 search uses one model seed to control runtime.
    # 050 refit should return to the 041 formal seed ensemble if search succeeds.
    OPTUNA_MODEL_SEED = 42

    USE_ORIGINAL = True

    # 041 baseline values for judgement.
    BASELINE_016_CV = 0.9519855866173828
    BASELINE_016_PUBLIC_LB = 0.95164
    BASELINE_041_CV = 0.9520124407043763
    BASELINE_041_PUBLIC_LB = 0.95173
    STRONG_CV_THRESHOLD = 0.95220

    # Optuna settings.
    N_TRIALS = 40
    TIMEOUT_SEC = 60 * 60 * 10
    OPTUNA_SEED = 42

    LOSSGUIDE_MAX_LEAVES = 64

    # Fixed parts of the 041 XGB configuration.
    # Search parameters are merged into this inside make_xgb_params().
    XGB_FIXED_PARAMS = {
        "n_estimators": 10000,
        "grow_policy": "lossguide",
        "max_depth": 0,
        "objective": "binary:logistic",
        "eval_metric": "auc",
        "tree_method": "hist",
        "device": "cuda",
        "n_jobs": -1,
        "enable_categorical": True,
        "early_stopping_rounds": 200,
    }

    # 041 feature flags are fixed.
    USE_SHARED001V2_FEATURES_041 = True
    KEEP_RAW_PITSTOP_041 = True
    ADD_RACE_COMPOUND_YEAR_CROSSES_041 = True
    ADD_039_LOG_FEATURES = False

    CLIP_LOW = 1e-6
    CLIP_HIGH = 1.0 - 1e-6

    VERBOSE = False


CFG.OUTDIR.mkdir(parents=True, exist_ok=True)


In [4]:
# ============================================================
# Utilities
# ============================================================

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


def print_section(title: str) -> None:
    print("\n" + "=" * 90)
    print(title)
    print("=" * 90)


def first_existing_dir(paths):
    for p in paths:
        path = Path(p)
        if (path / "train.csv").exists() and (path / "test.csv").exists():
            return path
    raise FileNotFoundError(f"No valid competition path found: {paths}")


def first_existing_file(paths, required=True):
    for p in paths:
        path = Path(p)
        if path.exists():
            return path
    if required:
        raise FileNotFoundError(f"No valid file path found: {paths}")
    return None


def resolve_target_column(df: pd.DataFrame) -> str:
    for c in CFG.TARGET_CANDIDATES:
        if c in df.columns:
            return c
    raise ValueError(f"No target column found among {CFG.TARGET_CANDIDATES}")


def clip_pred(x):
    return np.clip(x, CFG.CLIP_LOW, CFG.CLIP_HIGH)


def to_numpy(x):
    if hasattr(x, "get"):
        return x.get()
    return np.asarray(x)


def safe_colname(c):
    return (
        c.replace(" ", "_")
         .replace("(", "")
         .replace(")", "")
         .replace("/", "_")
         .replace("-", "_")
    )


def to_numeric_array(s):
    x = pd.to_numeric(s, errors="coerce").astype(float).values
    x = np.round(x, 6)
    return x


def round_to_step(s, step):
    return np.round(s / step) * step


seed_everything(CFG.SEED)


In [5]:
# ============================================================
# Load Data
# ============================================================

print_section("Load Data")

COMP_PATH = first_existing_dir(CFG.COMP_PATHS)

train = pd.read_csv(COMP_PATH / "train.csv")
test = pd.read_csv(COMP_PATH / "test.csv")
sample_submission = pd.read_csv(COMP_PATH / "sample_submission.csv")

target_col_train = resolve_target_column(train)
if target_col_train != CFG.TARGET:
    train = train.rename(columns={target_col_train: CFG.TARGET})

target_col_sub = resolve_target_column(sample_submission)
if target_col_sub != CFG.TARGET:
    sample_submission = sample_submission.rename(columns={target_col_sub: CFG.TARGET})

original_path = first_existing_file(CFG.ORIGINAL_PATHS, required=False)
orig = None

if CFG.USE_ORIGINAL and original_path is not None:
    orig = pd.read_csv(original_path)

    if CFG.TARGET not in orig.columns:
        target_col_orig = resolve_target_column(orig)
        orig = orig.rename(columns={target_col_orig: CFG.TARGET})

    if CFG.DANGER_COL in orig.columns:
        orig = orig.drop(columns=[CFG.DANGER_COL])
        print(f"Dropped danger column from original: {CFG.DANGER_COL}")

print("COMP_PATH:", COMP_PATH)
print("train:", train.shape)
print("test :", test.shape)
print("sample_submission:", sample_submission.shape)

if orig is not None:
    print("orig:", orig.shape)
    print("original path:", original_path)

print("target mean train:", train[CFG.TARGET].mean())
if orig is not None:
    print("target mean orig :", orig[CFG.TARGET].mean())

assert CFG.ID_COL in train.columns
assert CFG.ID_COL in test.columns
assert CFG.ID_COL in sample_submission.columns
assert CFG.TARGET in train.columns
assert CFG.TARGET in sample_submission.columns
assert test[CFG.ID_COL].equals(sample_submission[CFG.ID_COL])

if orig is not None:
    assert CFG.TARGET in orig.columns
    assert CFG.DANGER_COL not in orig.columns

train_ids = train[CFG.ID_COL].copy()
test_ids = test[CFG.ID_COL].copy()



Load Data
Dropped danger column from original: Normalized_TyreLife
COMP_PATH: /kaggle/input/competitions/playground-series-s6e5
train: (439140, 16)
test : (188165, 15)
sample_submission: (188165, 2)
orig: (101371, 15)
original path: /kaggle/input/datasets/aadigupta1601/f1-strategy-dataset-pit-stop-prediction/f1_strategy_dataset_v4.csv
target mean train: 0.19898210137996994
target mean orig : 0.25479673673930414


In [6]:
# ============================================================
# Base Columns
# ============================================================

TARGET = CFG.TARGET

BASE = [col for col in train.columns if col not in [CFG.ID_COL, TARGET]]
CATS = [col for col in BASE if train[col].dtype == "object"]

print(len(BASE), "Baseline Features.")
print(len(CATS), "Categorical Features.")
print("Categorical Columns:", CATS)

# shared_005 encoding:
# Driver / Compound / Race are mapped using train + test + original combined values.
# This is not target leakage, but it does use category vocabulary from test/orig.
for col in CATS:
    frames = [
        train[col].astype(str),
        test[col].astype(str),
    ]
    if orig is not None and col in orig.columns:
        frames.append(orig[col].astype(str))

    combined = pd.concat(frames, axis=0)
    uniques = combined.unique()
    mapping = {v: i for i, v in enumerate(uniques)}

    train[col] = train[col].astype(str).map(mapping).astype("int32").astype("category")
    test[col] = test[col].astype(str).map(mapping).astype("int32").astype("category")

    if orig is not None and col in orig.columns:
        orig[col] = orig[col].astype(str).map(mapping).astype("int32").astype("category")


14 Baseline Features.
3 Categorical Features.
Categorical Columns: ['Driver', 'Compound', 'Race']


In [7]:
# ============================================================
# Feature Engineering: NUM as CAT
# ============================================================

print_section("Feature Engineering: NUM as CAT")

NUM_as_CAT = []

NUM_CAT_BASE = [
    "LapTime (s)",
    "LapTime_Delta",
    "Cumulative_Degradation",
]

for c in NUM_CAT_BASE:
    new_col = f"{c}_cat"
    for df in [train, test] + ([orig] if orig is not None else []):
        if c in df.columns:
            df[new_col] = df[c].astype(str)
    NUM_as_CAT.append(new_col)

ROUND_CONFIG = {
    "LapTime (s)": {
        "round_digits": [1, 0],
        "round_steps": [0.5, 1.0, 2.0, 5.0],
    },
    "LapTime_Delta": {
        "round_digits": [1, 0],
        "round_steps": [0.5, 1.0, 2.0, 5.0, 10.0],
    },
    "Cumulative_Degradation": {
        "round_digits": [1, 0],
        "round_steps": [1.0, 2.0, 5.0, 10.0, 20.0],
    },
}

for c, cfg in ROUND_CONFIG.items():
    for d in cfg["round_digits"]:
        new_col = f"{c}_round{d}_cat"
        for df in [train, test] + ([orig] if orig is not None else []):
            if c in df.columns:
                df[new_col] = df[c].round(d).astype(str)
        NUM_as_CAT.append(new_col)

    for step in cfg["round_steps"]:
        step_name = str(step).replace(".", "p")
        new_col = f"{c}_round_step_{step_name}_cat"

        for df in [train, test] + ([orig] if orig is not None else []):
            if c in df.columns:
                df[new_col] = round_to_step(df[c], step).astype(str)

        NUM_as_CAT.append(new_col)

NUM_as_CAT = [c for c in NUM_as_CAT if c in train.columns and c in test.columns]
if orig is not None:
    NUM_as_CAT = [c for c in NUM_as_CAT if c in orig.columns]

print(len(NUM_as_CAT), "Total Num->Cat Features Created!")
print(NUM_as_CAT)



Feature Engineering: NUM as CAT
23 Total Num->Cat Features Created!
['LapTime (s)_cat', 'LapTime_Delta_cat', 'Cumulative_Degradation_cat', 'LapTime (s)_round1_cat', 'LapTime (s)_round0_cat', 'LapTime (s)_round_step_0p5_cat', 'LapTime (s)_round_step_1p0_cat', 'LapTime (s)_round_step_2p0_cat', 'LapTime (s)_round_step_5p0_cat', 'LapTime_Delta_round1_cat', 'LapTime_Delta_round0_cat', 'LapTime_Delta_round_step_0p5_cat', 'LapTime_Delta_round_step_1p0_cat', 'LapTime_Delta_round_step_2p0_cat', 'LapTime_Delta_round_step_5p0_cat', 'LapTime_Delta_round_step_10p0_cat', 'Cumulative_Degradation_round1_cat', 'Cumulative_Degradation_round0_cat', 'Cumulative_Degradation_round_step_1p0_cat', 'Cumulative_Degradation_round_step_2p0_cat', 'Cumulative_Degradation_round_step_5p0_cat', 'Cumulative_Degradation_round_step_10p0_cat', 'Cumulative_Degradation_round_step_20p0_cat']


In [8]:
# ============================================================
# Feature Engineering: DIGIT
# ============================================================

print_section("Feature Engineering: DIGIT")

DIGIT_FEATURES = []

DIGIT_BASE = [
    "Year",
    "PitStop",
    "LapNumber",
    "Stint",
    "TyreLife",
    "Position",
    "LapTime (s)",
    "LapTime_Delta",
    "Cumulative_Degradation",
    "RaceProgress",
    "Position_Change",
]

DECIMAL_DIGIT_BASE = [
    "LapTime (s)",
    "LapTime_Delta",
    "Cumulative_Degradation",
    "RaceProgress",
]

INT_POSITIONS = [1, 10, 100, 1000]
DECIMAL_POSITIONS = [1, 2, 3]

all_dfs = [train, test] + ([orig] if orig is not None else [])

for c in DIGIT_BASE:
    if not all(c in df.columns for df in all_dfs):
        print(f"[Skip] {c} is not found in all dataframes.")
        continue

    sc = safe_colname(c)

    sign_col = f"{sc}_sign"
    for df in all_dfs:
        x = to_numeric_array(df[c])
        sign = np.sign(np.nan_to_num(x, nan=0.0)).astype(np.int8)
        df[sign_col] = sign
    DIGIT_FEATURES.append(sign_col)

    for p in INT_POSITIONS:
        nc = f"{sc}_digit_{p}s"

        for df in all_dfs:
            x = to_numeric_array(df[c])
            x_abs = np.abs(np.nan_to_num(x, nan=0.0))
            int_part = np.floor(x_abs).astype(np.int64)
            digit = ((int_part // p) % 10).astype(np.int8)
            df[nc] = digit

        DIGIT_FEATURES.append(nc)

for c in DECIMAL_DIGIT_BASE:
    if not all(c in df.columns for df in all_dfs):
        print(f"[Skip] {c} is not found in all dataframes.")
        continue

    sc = safe_colname(c)

    for d in DECIMAL_POSITIONS:
        nc = f"{sc}_decimal_digit_{d}"

        for df in all_dfs:
            x = to_numeric_array(df[c])
            x_abs = np.abs(np.nan_to_num(x, nan=0.0))
            digit = (np.floor(x_abs * (10 ** d)).astype(np.int64) % 10).astype(np.int8)
            df[nc] = digit

        DIGIT_FEATURES.append(nc)

DIGIT_FEATURES = [c for c in DIGIT_FEATURES if c in train.columns and c in test.columns]
if orig is not None:
    DIGIT_FEATURES = [c for c in DIGIT_FEATURES if c in orig.columns]

print(len(DIGIT_FEATURES), "DIGIT Features Created!")
print(DIGIT_FEATURES[:30])



Feature Engineering: DIGIT
67 DIGIT Features Created!
['Year_sign', 'Year_digit_1s', 'Year_digit_10s', 'Year_digit_100s', 'Year_digit_1000s', 'PitStop_sign', 'PitStop_digit_1s', 'PitStop_digit_10s', 'PitStop_digit_100s', 'PitStop_digit_1000s', 'LapNumber_sign', 'LapNumber_digit_1s', 'LapNumber_digit_10s', 'LapNumber_digit_100s', 'LapNumber_digit_1000s', 'Stint_sign', 'Stint_digit_1s', 'Stint_digit_10s', 'Stint_digit_100s', 'Stint_digit_1000s', 'TyreLife_sign', 'TyreLife_digit_1s', 'TyreLife_digit_10s', 'TyreLife_digit_100s', 'TyreLife_digit_1000s', 'Position_sign', 'Position_digit_1s', 'Position_digit_10s', 'Position_digit_100s', 'Position_digit_1000s']


In [9]:
# ============================================================
# Feature Engineering: shared001v2-style features for 041
# ============================================================

print_section("Feature Engineering: shared001v2-style features for 041")

SHARED001V2_FLOOR_CAT_COLS_041 = [
    "PitStop",
    "LapNumber",
    "Stint",
    "TyreLife",
    "Position",
    "LapTime (s)",
    "LapTime_Delta",
    "Cumulative_Degradation",
    "RaceProgress",
    "Position_Change",
]

SHARED001V2_BIN_COLS_041 = [
    "RaceProgress_200_quantile_bin_",
    "LapTime (s)_7_quantile_bin_",
]

SHARED001V2_CROSS_COLS_041 = [
    "Race_Compound_",
    "Race_Year_",
]

SHARED001V2_COUNT_SOURCE_COLS_041 = [
    "PitStop_cat_",
    "RaceProgress_200_quantile_bin_",
    "LapTime (s)_7_quantile_bin_",
    "Race_Compound_",
    "Race_Year_",
    "Compound",
    "Race",
    "Driver",
]

SHARED001V2_CAT_FEATURES_041 = []
SHARED001V2_COUNT_FEATURES_041 = []


def _safe_qbin_041(s: pd.Series, q: int, name: str) -> pd.Series:
    """Robust target-free quantile binning fitted on the combined unsupervised table."""
    x = pd.to_numeric(s, errors="coerce")
    if x.notna().sum() == 0:
        return pd.Series([f"{name}_missing"] * len(s), index=s.index, dtype="object")

    r = x.rank(method="first")
    try:
        b = pd.qcut(r, q=q, labels=False, duplicates="drop")
    except Exception:
        b = pd.cut(r, bins=min(q, max(int(r.nunique()), 1)), labels=False)

    return (
        pd.Series(b, index=s.index)
        .astype("Int64")
        .astype(str)
        .replace("<NA>", f"{name}_missing")
        .map(lambda z: f"{name}_{z}")
        .astype(str)
    )


def _build_shared001v2_features_041(all_df: pd.DataFrame) -> pd.DataFrame:
    """Add 029/038-style target-free representation features.

    This block intentionally does not add the 039 LOG features.
    """
    out = all_df.copy()

    if not CFG.USE_SHARED001V2_FEATURES_041:
        return out

    if "PitStop" in out.columns:
        pit = pd.to_numeric(out["PitStop"], errors="coerce").fillna(-1)
        out["PitStop_cat_"] = np.floor(pit).astype(np.int16).astype(str)

    for col in SHARED001V2_FLOOR_CAT_COLS_041:
        if col not in out.columns:
            continue

        x = pd.to_numeric(out[col], errors="coerce")
        cat_col = f"{col}_cat_"
        out[cat_col] = (
            np.floor(x.fillna(-9999))
            .astype(np.int32)
            .astype(str)
        )

        if cat_col not in SHARED001V2_COUNT_SOURCE_COLS_041:
            SHARED001V2_COUNT_SOURCE_COLS_041.append(cat_col)

    if "RaceProgress" in out.columns:
        out["RaceProgress_200_quantile_bin_"] = _safe_qbin_041(
            out["RaceProgress"], q=200, name="RP200"
        )

    if "LapTime (s)" in out.columns:
        out["LapTime (s)_7_quantile_bin_"] = _safe_qbin_041(
            out["LapTime (s)"], q=7, name="LT7"
        )

    if CFG.ADD_RACE_COMPOUND_YEAR_CROSSES_041:
        if {"Race", "Compound"}.issubset(out.columns):
            out["Race_Compound_"] = (
                out["Race"].astype(str) + "__" + out["Compound"].astype(str)
            )

        if {"Race", "Year"}.issubset(out.columns):
            out["Race_Year_"] = (
                out["Race"].astype(str) + "__" + out["Year"].astype(str)
            )

    count_cols = [
        c for c in SHARED001V2_COUNT_SOURCE_COLS_041
        if c in out.columns
    ]

    for col in count_cols:
        vc = out[col].astype(str).value_counts(dropna=False)

        if col == "PitStop_cat_":
            count_name = "_PitStop_cat_count"
        else:
            count_name = f"{col}_count_041"

        out[count_name] = (
            out[col].astype(str).map(vc).fillna(0).astype(np.float32)
        )

    return out


if CFG.USE_SHARED001V2_FEATURES_041:
    # Fit the target-free bin/count vocabulary on the combined train/test/original table.
    # This follows the same transductive availability as frequency/count encoding.
    n_train = len(train)
    n_test = len(test)
    n_orig = len(orig) if orig is not None else 0

    train_feat = train.drop(columns=[TARGET], errors="ignore").copy()
    test_feat = test.copy()

    frames = [train_feat, test_feat]

    if orig is not None:
        orig_feat = orig.drop(columns=[TARGET], errors="ignore").copy()
        frames.append(orig_feat)

    combined_feat = pd.concat(frames, axis=0, ignore_index=True, sort=False)
    combined_feat = _build_shared001v2_features_041(combined_feat)

    train_added = combined_feat.iloc[:n_train].reset_index(drop=True)
    test_added = combined_feat.iloc[n_train:n_train + n_test].reset_index(drop=True)

    for col in train_added.columns:
        if col not in train.columns:
            train[col] = train_added[col].values
        if col not in test.columns:
            test[col] = test_added[col].values

    if orig is not None:
        orig_added = combined_feat.iloc[n_train + n_test:n_train + n_test + n_orig].reset_index(drop=True)
        for col in orig_added.columns:
            if col not in orig.columns:
                orig[col] = orig_added[col].values

    for col in [
        "PitStop_cat_",
        "RaceProgress_200_quantile_bin_",
        "LapTime (s)_7_quantile_bin_",
        "Race_Compound_",
        "Race_Year_",
    ]:
        if col in train.columns and col in test.columns:
            SHARED001V2_CAT_FEATURES_041.append(col)

    for col in SHARED001V2_FLOOR_CAT_COLS_041:
        cat_col = f"{col}_cat_"
        if cat_col in train.columns and cat_col in test.columns:
            SHARED001V2_CAT_FEATURES_041.append(cat_col)

    SHARED001V2_CAT_FEATURES_041 = list(dict.fromkeys(SHARED001V2_CAT_FEATURES_041))

    for col in train.columns:
        if col == "_PitStop_cat_count" or col.endswith("_count_041"):
            if col in test.columns:
                SHARED001V2_COUNT_FEATURES_041.append(col)

    SHARED001V2_COUNT_FEATURES_041 = list(dict.fromkeys(SHARED001V2_COUNT_FEATURES_041))

    # Treat the added categorical/bin views like 016's NUM_as_CAT block:
    # they receive single-column frequency encoding and single-column TE.
    for col in SHARED001V2_CAT_FEATURES_041:
        if col not in NUM_as_CAT:
            NUM_as_CAT.append(col)

    # XGBoost categorical support requires category dtype for non-TE object columns.
    for df in [train, test] + ([orig] if orig is not None else []):
        for col in SHARED001V2_CAT_FEATURES_041:
            if col in df.columns:
                df[col] = df[col].astype(str).astype("category")

print("shared001v2 cat features 041:", len(SHARED001V2_CAT_FEATURES_041))
print(SHARED001V2_CAT_FEATURES_041)
print("shared001v2 count features 041:", len(SHARED001V2_COUNT_FEATURES_041))
print(SHARED001V2_COUNT_FEATURES_041)

assert not CFG.ADD_039_LOG_FEATURES, "041 must not add 039 LOG features."



Feature Engineering: shared001v2-style features for 041
shared001v2 cat features 041: 14
['PitStop_cat_', 'RaceProgress_200_quantile_bin_', 'LapTime (s)_7_quantile_bin_', 'Race_Compound_', 'Race_Year_', 'LapNumber_cat_', 'Stint_cat_', 'TyreLife_cat_', 'Position_cat_', 'LapTime (s)_cat_', 'LapTime_Delta_cat_', 'Cumulative_Degradation_cat_', 'RaceProgress_cat_', 'Position_Change_cat_']
shared001v2 count features 041: 17
['_PitStop_cat_count', 'RaceProgress_200_quantile_bin__count_041', 'LapTime (s)_7_quantile_bin__count_041', 'Race_Compound__count_041', 'Race_Year__count_041', 'Compound_count_041', 'Race_count_041', 'Driver_count_041', 'LapNumber_cat__count_041', 'Stint_cat__count_041', 'TyreLife_cat__count_041', 'Position_cat__count_041', 'LapTime (s)_cat__count_041', 'LapTime_Delta_cat__count_041', 'Cumulative_Degradation_cat__count_041', 'RaceProgress_cat__count_041', 'Position_Change_cat__count_041']


In [10]:
# ============================================================
# Model Feature Configuration
# ============================================================

print_section("Model Feature Configuration")

TE_BASE = [
    "Driver",
    "Compound",
    "Race",
    "Year",
    "PitStop",
    "LapNumber",
    "Stint",
    "TyreLife",
    "Position",
    "RaceProgress",
    "Position_Change",
]

TE_BASE = [c for c in TE_BASE if c in train.columns and c in test.columns]
if orig is not None:
    TE_BASE = [c for c in TE_BASE if c in orig.columns]

BIGRAM_SPECS = list(combinations(TE_BASE, 2))

FEATURES = BASE + NUM_as_CAT + DIGIT_FEATURES + SHARED001V2_COUNT_FEATURES_041
FEATURES = [c for c in FEATURES if c in train.columns and c in test.columns]
if orig is not None:
    FEATURES = [c for c in FEATURES if c in orig.columns]

# Deduplicate while preserving order
FEATURES = list(dict.fromkeys(FEATURES))

print(len(BIGRAM_SPECS), "BIGRAM specs")
print(len(FEATURES), "Base features")
print(len(SHARED001V2_CAT_FEATURES_041), "shared001v2 cat features 041")
print(len(SHARED001V2_COUNT_FEATURES_041), "shared001v2 count features 041")
print("FEATURES sample:", FEATURES[:30])

assert CFG.DANGER_COL not in FEATURES



Model Feature Configuration
55 BIGRAM specs
135 Base features
14 shared001v2 cat features 041
17 shared001v2 count features 041
FEATURES sample: ['Driver', 'Compound', 'Race', 'Year', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position', 'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation', 'RaceProgress', 'Position_Change', 'LapTime (s)_cat', 'LapTime_Delta_cat', 'Cumulative_Degradation_cat', 'LapTime (s)_round1_cat', 'LapTime (s)_round0_cat', 'LapTime (s)_round_step_0p5_cat', 'LapTime (s)_round_step_1p0_cat', 'LapTime (s)_round_step_2p0_cat', 'LapTime (s)_round_step_5p0_cat', 'LapTime_Delta_round1_cat', 'LapTime_Delta_round0_cat', 'LapTime_Delta_round_step_0p5_cat', 'LapTime_Delta_round_step_1p0_cat', 'LapTime_Delta_round_step_2p0_cat', 'LapTime_Delta_round_step_5p0_cat', 'LapTime_Delta_round_step_10p0_cat']


In [11]:
# ============================================================
# Encoding Helpers
# ============================================================

def make_inner_fold_ids(y, n_splits=5, seed=42):
    fold_ids = np.zeros(len(y), dtype=np.int32)

    inner = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=seed,
    )

    dummy_X = np.zeros((len(y), 1))

    for fold, (_, va_idx) in enumerate(inner.split(dummy_X, y)):
        fold_ids[va_idx] = fold

    return cudf.Series(fold_ids)


def add_frequency_encode(
    X_tr,
    X_va,
    X_test,
    cols,
    out_col=None,
    normalize=True,
):
    cols = list(cols)

    if out_col is None:
        out_col = "fe__" + "__".join(cols)

    X_tr_key = X_tr[cols].astype(str)
    X_va_key = X_va[cols].astype(str)
    X_test_key = X_test[cols].astype(str)

    denom = len(X_tr) if normalize else 1.0

    if len(cols) == 1:
        c = cols[0]
        freq_map = X_tr_key[c].value_counts(dropna=False)

        X_tr[out_col] = (
            X_tr_key[c]
            .map(freq_map)
            .fillna(0)
            .astype(np.float32)
            .values / denom
        )

        X_va[out_col] = (
            X_va_key[c]
            .map(freq_map)
            .fillna(0)
            .astype(np.float32)
            .values / denom
        )

        X_test[out_col] = (
            X_test_key[c]
            .map(freq_map)
            .fillna(0)
            .astype(np.float32)
            .values / denom
        )

    else:
        freq_df = (
            X_tr_key
            .groupby(cols, dropna=False)
            .size()
            .reset_index(name=out_col)
        )

        def map_joint_freq(df_key):
            return (
                df_key
                .reset_index(drop=True)
                .merge(freq_df, on=cols, how="left")[out_col]
                .fillna(0)
                .astype(np.float32)
                .values / denom
            )

        X_tr[out_col] = map_joint_freq(X_tr_key)
        X_va[out_col] = map_joint_freq(X_va_key)
        X_test[out_col] = map_joint_freq(X_test_key)


def add_cuml_te(
    X_tr,
    X_va,
    X_test,
    X_tr_g,
    X_va_g,
    X_test_g,
    y_tr_g,
    cols,
    out_col,
    fold_ids_g,
    seed=42,
    smooth=20,
    n_folds=5,
):
    cols = list(cols)

    te = cuTargetEncoder(
        n_folds=n_folds,
        smooth=smooth,
        seed=seed,
        split_method="random",
        output_type="numpy",
        stat="mean",
        multi_feature_mode="combination",
    )

    tr_enc = te.fit_transform(
        X_tr_g[cols],
        y_tr_g,
        fold_ids=fold_ids_g,
    )

    va_enc = te.transform(X_va_g[cols])
    test_enc = te.transform(X_test_g[cols])

    X_tr[out_col] = to_numpy(tr_enc).reshape(-1).astype(np.float32)
    X_va[out_col] = to_numpy(va_enc).reshape(-1).astype(np.float32)
    X_test[out_col] = to_numpy(test_enc).reshape(-1).astype(np.float32)


In [12]:
# ============================================================
# Build Fold Matrices Once for Optuna
# ============================================================

print_section("Build Fold Matrices Once for Optuna")

fold_data = []
fold_build_records = []
all_feature_cols_after_fe_te = None

skf = StratifiedKFold(
    n_splits=CFG.N_SPLITS,
    shuffle=True,
    random_state=CFG.SEED,
)

if orig is not None and CFG.USE_ORIGINAL:
    X_orig = orig[FEATURES].copy()
    y_orig = orig[TARGET].copy().reset_index(drop=True)
else:
    X_orig = None
    y_orig = None

for fold, (tr_idx, va_idx) in enumerate(skf.split(train[FEATURES], train[TARGET]), 1):
    print("\n" + "=" * 90)
    print(f"Prepare Fold {fold}/{CFG.N_SPLITS}")
    print("=" * 90)

    X_tr_base = train[FEATURES].iloc[tr_idx].copy().reset_index(drop=True)
    y_tr_base = train[TARGET].iloc[tr_idx].copy().reset_index(drop=True)

    X_va = train[FEATURES].iloc[va_idx].copy().reset_index(drop=True)
    y_va = train[TARGET].iloc[va_idx].copy().reset_index(drop=True)

    # X_test is needed only because the existing FE/TE helper functions mutate train/valid/test together.
    X_test_dummy = test[FEATURES].copy().reset_index(drop=True)

    if X_orig is not None and CFG.USE_ORIGINAL:
        X_tr = pd.concat(
            [X_tr_base.reset_index(drop=True), X_orig.reset_index(drop=True)],
            axis=0,
            ignore_index=True,
        )
        y_tr = pd.concat(
            [y_tr_base.reset_index(drop=True), y_orig.reset_index(drop=True)],
            axis=0,
            ignore_index=True,
        )
    else:
        X_tr = X_tr_base.copy()
        y_tr = y_tr_base.copy()

    print("Base:", X_tr.shape, X_va.shape, X_test_dummy.shape)
    print("target mean train fold:", float(y_tr.mean()))
    print("target mean valid fold:", float(y_va.mean()))

    te_source_cols = sorted(set(["Driver"] + NUM_as_CAT + TE_BASE))
    te_source_cols = [c for c in te_source_cols if c in X_tr.columns]

    X_tr_g = cudf.from_pandas(X_tr[te_source_cols].astype(str))
    X_va_g = cudf.from_pandas(X_va[te_source_cols].astype(str))
    X_test_g = cudf.from_pandas(X_test_dummy[te_source_cols].astype(str))
    y_tr_g = cudf.Series(y_tr.values)

    te_seed = 1000 + fold
    fold_ids_g = make_inner_fold_ids(
        y_tr,
        n_splits=CFG.N_TE_FOLDS,
        seed=te_seed,
    )

    # -------------------------
    # Frequency Encoding
    # -------------------------

    FE_SINGLE_COLS = sorted(set(["Driver"] + NUM_as_CAT))
    FE_SINGLE_COLS = [c for c in FE_SINGLE_COLS if c in X_tr.columns]

    for c in FE_SINGLE_COLS:
        add_frequency_encode(
            X_tr,
            X_va,
            X_test_dummy,
            cols=(c,),
            out_col=f"fe__{c}",
            normalize=True,
        )

    for cols in BIGRAM_SPECS:
        if all(c in X_tr.columns for c in cols):
            add_frequency_encode(
                X_tr,
                X_va,
                X_test_dummy,
                cols=cols,
                out_col="fe2__" + "__".join(cols),
                normalize=True,
            )

    print("After FE:", X_tr.shape, X_va.shape, X_test_dummy.shape)

    # -------------------------
    # Target Encoding
    # -------------------------

    if "Driver" in X_tr.columns:
        add_cuml_te(
            X_tr,
            X_va,
            X_test_dummy,
            X_tr_g,
            X_va_g,
            X_test_g,
            y_tr_g,
            cols=("Driver",),
            out_col="te_Driver",
            fold_ids_g=fold_ids_g,
            seed=te_seed,
            smooth=CFG.TE_SMOOTH,
            n_folds=CFG.N_TE_FOLDS,
        )

    for c in NUM_as_CAT:
        if c in X_tr.columns:
            add_cuml_te(
                X_tr,
                X_va,
                X_test_dummy,
                X_tr_g,
                X_va_g,
                X_test_g,
                y_tr_g,
                cols=(c,),
                out_col=c,
                fold_ids_g=fold_ids_g,
                seed=te_seed,
                smooth=CFG.TE_SMOOTH,
                n_folds=CFG.N_TE_FOLDS,
            )

    for cols in BIGRAM_SPECS:
        if all(c in X_tr.columns for c in cols):
            add_cuml_te(
                X_tr,
                X_va,
                X_test_dummy,
                X_tr_g,
                X_va_g,
                X_test_g,
                y_tr_g,
                cols=cols,
                out_col="te2__" + "__".join(cols),
                fold_ids_g=fold_ids_g,
                seed=te_seed,
                smooth=CFG.TE_SMOOTH,
                n_folds=CFG.N_TE_FOLDS,
            )

    print("After TE:", X_tr.shape, X_va.shape, X_test_dummy.shape)

    if all_feature_cols_after_fe_te is None:
        all_feature_cols_after_fe_te = list(X_tr.columns)
    else:
        assert list(X_tr.columns) == all_feature_cols_after_fe_te, "Feature columns differ across folds"

    fold_data.append({
        "fold": fold,
        "tr_idx": tr_idx,
        "va_idx": va_idx,
        "X_tr": X_tr,
        "y_tr": y_tr,
        "X_va": X_va,
        "y_va": y_va,
    })

    fold_build_records.append({
        "fold": fold,
        "n_train_comp": int(len(X_tr_base)),
        "n_train_orig": int(len(X_orig)) if X_orig is not None and CFG.USE_ORIGINAL else 0,
        "n_train_total": int(len(X_tr)),
        "n_valid": int(len(X_va)),
        "n_features_after_fe_te": int(X_tr.shape[1]),
        "target_mean_train": float(y_tr.mean()),
        "target_mean_valid": float(y_va.mean()),
    })

    del X_tr_base, y_tr_base, X_test_dummy
    del X_tr_g, X_va_g, X_test_g, y_tr_g, fold_ids_g
    gc.collect()

fold_build_df = pd.DataFrame(fold_build_records)
print("Prepared fold matrices:", len(fold_data))
display(fold_build_df)



Build Fold Matrices Once for Optuna

Prepare Fold 1/5
Base: (452683, 135) (87828, 135) (188165, 135)
target mean train fold: 0.21147911452385001
target mean valid fold: 0.19899121009245344
After FE: (452683, 228) (87828, 228) (188165, 228)
After TE: (452683, 284) (87828, 284) (188165, 284)

Prepare Fold 2/5
Base: (452683, 135) (87828, 135) (188165, 135)
target mean train fold: 0.2114813235752171
target mean valid fold: 0.19897982420184906
After FE: (452683, 228) (87828, 228) (188165, 228)
After TE: (452683, 284) (87828, 284) (188165, 284)

Prepare Fold 3/5
Base: (452683, 135) (87828, 135) (188165, 135)
target mean train fold: 0.2114813235752171
target mean valid fold: 0.19897982420184906
After FE: (452683, 228) (87828, 228) (188165, 228)
After TE: (452683, 284) (87828, 284) (188165, 284)

Prepare Fold 4/5
Base: (452683, 135) (87828, 135) (188165, 135)
target mean train fold: 0.2114813235752171
target mean valid fold: 0.19897982420184906
After FE: (452683, 228) (87828, 228) (188165, 22

,fold,n_train_comp,n_train_orig,n_train_total,n_valid,n_features_after_fe_te,target_mean_train,target_mean_valid
0,1,351312,101371,452683,87828,284,0.211479,0.198991
1,2,351312,101371,452683,87828,284,0.211481,0.198980
2,3,351312,101371,452683,87828,284,0.211481,0.198980
3,4,351312,101371,452683,87828,284,0.211481,0.198980
4,5,351312,101371,452683,87828,284,0.211481,0.198980


In [13]:
# ============================================================
# Optuna Search
# ============================================================

print_section("Optuna Search")


def suggest_xgb_params(trial):
    return {
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.06, log=True),
        "max_leaves": trial.suggest_int("max_leaves", 32, 128),
        "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 30.0, log=True),
        "subsample": trial.suggest_float("subsample", 0.65, 0.95),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.65, 0.95),
        "reg_alpha": trial.suggest_float("reg_alpha", 1.0e-4, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1.0, 50.0, log=True),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "max_bin": trial.suggest_categorical("max_bin", [128, 256]),
    }


def make_xgb_params(search_params, seed):
    params = CFG.XGB_FIXED_PARAMS.copy()
    params.update(search_params)
    params["random_state"] = int(seed)
    return params


def objective(trial):
    search_params = suggest_xgb_params(trial)

    fold_aucs = []
    fold_best_iters = []

    for item in fold_data:
        fold = item["fold"]
        X_tr = item["X_tr"]
        y_tr = item["y_tr"]
        X_va = item["X_va"]
        y_va = item["y_va"]

        model_seed = CFG.OPTUNA_MODEL_SEED + (fold - 1) * 100
        params = make_xgb_params(search_params, model_seed)

        model = XGBClassifier(**params)
        model.fit(
            X_tr,
            y_tr,
            eval_set=[(X_va, y_va)],
            verbose=CFG.VERBOSE,
        )

        va_pred = clip_pred(model.predict_proba(X_va)[:, 1]).astype(np.float32)
        auc = roc_auc_score(y_va, va_pred)
        fold_aucs.append(float(auc))

        try:
            best_iter = int(model.best_iteration)
        except Exception:
            best_iter = -1
        fold_best_iters.append(best_iter)

        del model, va_pred
        gc.collect()

    mean_auc = float(np.mean(fold_aucs))
    std_auc = float(np.std(fold_aucs))

    trial.set_user_attr("fold_aucs", fold_aucs)
    trial.set_user_attr("fold_best_iters", fold_best_iters)
    trial.set_user_attr("mean_best_iter", float(np.mean(fold_best_iters)))
    trial.set_user_attr("std_auc", std_auc)

    print(
        f"Trial {trial.number}: mean_auc={mean_auc:.9f}, std={std_auc:.9f}, "
        f"params={search_params}"
    )

    return mean_auc


sampler = optuna.samplers.TPESampler(seed=CFG.OPTUNA_SEED)
study = optuna.create_study(direction="maximize", sampler=sampler)

study.optimize(
    objective,
    n_trials=CFG.N_TRIALS,
    timeout=CFG.TIMEOUT_SEC,
    show_progress_bar=True,
)

best_trial = study.best_trial
best_params = dict(best_trial.params)
best_cv_auc = float(best_trial.value)

print("Best trial:", best_trial.number)
print("Best CV AUC:", best_cv_auc)
print("Best params:")
print(json.dumps(best_params, indent=2))

trials_df = study.trials_dataframe(attrs=("number", "value", "params", "state", "datetime_start", "datetime_complete", "duration", "user_attrs"))
display(trials_df.sort_values("value", ascending=False).head(20))


[I 2026-05-23 01:03:08,615] A new study created in memory with name: no-name-177e574b-060e-4b0f-934b-1d305be4bd7a



Optuna Search


  0%|          | 0/40 [00:00<?, ?it/s]

Trial 0: mean_auc=0.951760035, std=0.000743866, params={'learning_rate': 0.01956360392823701, 'max_leaves': 124, 'min_child_weight': 12.057126287443763, 'subsample': 0.829597545259111, 'colsample_bytree': 0.696805592132731, 'reg_alpha': 0.000602521573620386, 'reg_lambda': 1.2551115172973832, 'gamma': 4.330880728874676, 'max_bin': 256}
[I 2026-05-23 01:11:30,880] Trial 0 finished with value: 0.9517600346541251 and parameters: {'learning_rate': 0.01956360392823701, 'max_leaves': 124, 'min_child_weight': 12.057126287443763, 'subsample': 0.829597545259111, 'colsample_bytree': 0.696805592132731, 'reg_alpha': 0.000602521573620386, 'reg_lambda': 1.2551115172973832, 'gamma': 4.330880728874676, 'max_bin': 256}. Best is trial 0 with value: 0.9517600346541251.
Trial 1: mean_auc=0.952137391, std=0.000779581, params={'learning_rate': 0.010375710602388847, 'max_leaves': 126, 'min_child_weight': 16.967533607196547, 'subsample': 0.7137017332034828, 'colsample_bytree': 0.7045474901621303, 'reg_alpha': 

,number,value,params_colsample_bytree,params_gamma,params_learning_rate,params_max_bin,params_max_leaves,params_min_child_weight,params_reg_alpha,params_reg_lambda,params_subsample,state,datetime_start,datetime_complete,duration,user_attrs_fold_aucs,user_attrs_fold_best_iters,user_attrs_mean_best_iter,user_attrs_std_auc
11,11,0.952227,0.658599,1.789675,0.010068,128,128,7.209286,0.000124,40.207068,0.658275,COMPLETE,2026-05-23 02:53:29.383383,2026-05-23 03:12:05.100115,0 days 00:18:35.716732,"[0.9534615032906542, 0.9510947980844374, 0.952...","[4392, 4283, 3469, 4214, 4635]",4198.6,0.000852
35,35,0.952221,0.668237,2.815019,0.010923,128,123,1.346891,0.018718,7.735375,0.667577,COMPLETE,2026-05-23 08:20:09.469752,2026-05-23 08:36:27.323949,0 days 00:16:17.854197,"[0.9534468874769253, 0.9511475666905062, 0.952...","[4699, 4153, 4048, 4683, 4730]",4462.6,0.000823
10,10,0.952219,0.654889,1.626800,0.010375,128,122,7.190751,0.000156,42.719062,0.655389,COMPLETE,2026-05-23 02:34:54.449910,2026-05-23 02:53:29.370898,0 days 00:18:34.920988,"[0.9534160316442416, 0.9511291034735547, 0.952...","[4838, 3918, 4161, 4075, 4522]",4302.8,0.000827
27,27,0.952215,0.696661,0.734944,0.011423,128,120,2.181620,0.003350,13.394911,0.680485,COMPLETE,2026-05-23 06:42:25.929108,2026-05-23 06:57:33.663494,0 days 00:15:07.734386,"[0.9535585001435681, 0.9511788114150688, 0.951...","[5051, 3742, 3225, 3438, 3982]",3887.6,0.000864
28,28,0.952185,0.698739,0.546845,0.011245,128,121,1.745944,0.002695,6.974141,0.675165,COMPLETE,2026-05-23 06:57:33.675090,2026-05-23 07:12:25.818618,0 days 00:14:52.143528,"[0.9535251856366739, 0.9510984602269187, 0.951...","[4455, 3827, 2898, 4123, 4512]",3963.0,0.000866
32,32,0.952182,0.704706,1.090063,0.011397,128,111,1.601143,0.000691,12.369631,0.670518,COMPLETE,2026-05-23 07:44:16.991578,2026-05-23 08:00:21.604466,0 days 00:16:04.612888,"[0.9534421246597974, 0.9510365181674413, 0.951...","[4378, 3931, 4090, 4383, 5163]",4389.0,0.000874
37,37,0.952180,0.725478,2.946876,0.010630,128,128,1.311340,0.015993,2.442471,0.688032,COMPLETE,2026-05-23 08:46:20.622420,2026-05-23 09:01:33.378797,0 days 00:15:12.756377,"[0.9534662819675723, 0.9512242085614356, 0.951...","[4920, 4795, 3247, 4311, 4210]",4296.6,0.000823
13,13,0.952162,0.651355,1.561175,0.010176,128,81,6.293997,0.000134,49.591613,0.708163,COMPLETE,2026-05-23 03:24:39.679047,2026-05-23 03:43:45.330159,0 days 00:19:05.651112,"[0.9534021730338506, 0.950996264694378, 0.9519...","[5853, 4549, 5250, 6236, 6342]",5646.0,0.000859
24,24,0.952159,0.678641,1.283166,0.011191,128,119,3.641202,0.001605,26.075313,0.670475,COMPLETE,2026-05-23 05:58:12.498957,2026-05-23 06:14:01.597309,0 days 00:15:49.098352,"[0.9533955294082173, 0.9510377979855852, 0.951...","[4011, 3878, 3430, 4081, 3796]",3839.2,0.000841
22,22,0.952158,0.675186,1.047695,0.011531,128,108,6.054787,0.000328,46.549155,0.714216,COMPLETE,2026-05-23 05:28:11.046968,2026-05-23 05:44:46.437085,0 days 00:16:35.390117,"[0.9534785684253835, 0.9510891582630127, 0.951...","[4617, 4053, 3491, 4327, 4174]",4132.4,0.000842


In [14]:
# ============================================================
# Save Optuna Artifacts
# ============================================================

print_section("Save Optuna Artifacts")

best_payload = {
    "experiment_id": CFG.EXP_ID,
    "base_experiment": "exp_20260520_041_xgb_shared001v2_features_min",
    "created_at": CFG.CREATED_AT,
    "best_trial_number": int(best_trial.number),
    "best_cv_auc": float(best_cv_auc),
    "baseline_016_cv_auc": float(CFG.BASELINE_016_CV),
    "baseline_041_cv_auc": float(CFG.BASELINE_041_CV),
    "delta_vs_041_cv": float(best_cv_auc - CFG.BASELINE_041_CV),
    "strong_cv_threshold": float(CFG.STRONG_CV_THRESHOLD),
    "best_params": best_params,
    "effective_xgb_params_for_search_seed": make_xgb_params(best_params, CFG.OPTUNA_MODEL_SEED),
    "fixed_xgb_params": CFG.XGB_FIXED_PARAMS,
    "optuna": {
        "direction": "maximize",
        "n_trials_requested": int(CFG.N_TRIALS),
        "n_trials_completed": int(len(study.trials)),
        "timeout_sec": int(CFG.TIMEOUT_SEC) if CFG.TIMEOUT_SEC is not None else None,
        "sampler": "TPESampler",
        "seed": int(CFG.OPTUNA_SEED),
    },
    "search_design": {
        "feature_set": "same_as_041",
        "folds": "same_as_041",
        "original_rows": "same_as_041",
        "model_seed_policy": "single seed for search only",
        "optuna_model_seed": int(CFG.OPTUNA_MODEL_SEED),
        "no_new_features": True,
        "add_039_log_features": False,
    },
}

with open(CFG.OUTDIR / "best_params_049.json", "w") as f:
    json.dump(best_payload, f, indent=2)

trials_df.to_csv(CFG.OUTDIR / "optuna_trials_049.csv", index=False)

study_summary = {
    "experiment_id": CFG.EXP_ID,
    "best_trial_number": int(best_trial.number),
    "best_cv_auc": float(best_cv_auc),
    "best_params": best_params,
    "n_trials_completed": int(len(study.trials)),
    "baseline_041_cv_auc": float(CFG.BASELINE_041_CV),
    "delta_vs_041_cv": float(best_cv_auc - CFG.BASELINE_041_CV),
    "decision_hint": (
        "success_go_to_050_refit"
        if best_cv_auc > CFG.BASELINE_041_CV
        else "weak_result_do_not_deepen_xgb_optuna"
    ),
}
with open(CFG.OUTDIR / "optuna_study_summary_049.json", "w") as f:
    json.dump(study_summary, f, indent=2)

with open(CFG.OUTDIR / "optuna_study_049.pkl", "wb") as f:
    pickle.dump(study, f)

# Feature columns after FE/TE
feature_df = pd.DataFrame({
    "feature": all_feature_cols_after_fe_te,
    "is_base": [c in BASE for c in all_feature_cols_after_fe_te],
    "is_num_as_cat": [c in NUM_as_CAT for c in all_feature_cols_after_fe_te],
    "is_digit": [c in DIGIT_FEATURES for c in all_feature_cols_after_fe_te],
    "is_shared001v2_cat_041": [c in SHARED001V2_CAT_FEATURES_041 for c in all_feature_cols_after_fe_te],
    "is_shared001v2_count_041": [c in SHARED001V2_COUNT_FEATURES_041 for c in all_feature_cols_after_fe_te],
    "is_frequency": [c.startswith("fe__") or c.startswith("fe2__") for c in all_feature_cols_after_fe_te],
    "is_target_encoding": [c.startswith("te_") or c.startswith("te2__") for c in all_feature_cols_after_fe_te],
    "contains_year": ["Year" in c for c in all_feature_cols_after_fe_te],
    "contains_race": ["Race" in c for c in all_feature_cols_after_fe_te],
    "contains_driver": ["Driver" in c for c in all_feature_cols_after_fe_te],
    "contains_pitstop": ["PitStop" in c for c in all_feature_cols_after_fe_te],
})
feature_df.to_csv(CFG.OUTDIR / "feature_columns.csv", index=False)

pd.Series(FEATURES, name="base_feature").to_csv(CFG.OUTDIR / "base_features.csv", index=False)
pd.Series(NUM_as_CAT, name="num_as_cat_feature").to_csv(CFG.OUTDIR / "num_as_cat_features.csv", index=False)
pd.Series(DIGIT_FEATURES, name="digit_feature").to_csv(CFG.OUTDIR / "digit_features.csv", index=False)

shared001v2_df = pd.DataFrame({
    "feature": SHARED001V2_CAT_FEATURES_041 + SHARED001V2_COUNT_FEATURES_041,
    "kind": ["cat_or_bin"] * len(SHARED001V2_CAT_FEATURES_041) + ["count"] * len(SHARED001V2_COUNT_FEATURES_041),
})
shared001v2_df.to_csv(CFG.OUTDIR / "shared001v2_features_049.csv", index=False)

fold_build_df.to_csv(CFG.OUTDIR / "fold_build_records.csv", index=False)

print("Saved:")
for name in [
    "best_params_049.json",
    "optuna_trials_049.csv",
    "optuna_study_summary_049.json",
    "optuna_study_049.pkl",
    "feature_columns.csv",
    "shared001v2_features_049.csv",
    "fold_build_records.csv",
]:
    print(" -", CFG.OUTDIR / name)



Save Optuna Artifacts
Saved:
 - /kaggle/working/exp_20260523_049_xgb_shared001v2_features_optuna_search_min/best_params_049.json
 - /kaggle/working/exp_20260523_049_xgb_shared001v2_features_optuna_search_min/optuna_trials_049.csv
 - /kaggle/working/exp_20260523_049_xgb_shared001v2_features_optuna_search_min/optuna_study_summary_049.json
 - /kaggle/working/exp_20260523_049_xgb_shared001v2_features_optuna_search_min/optuna_study_049.pkl
 - /kaggle/working/exp_20260523_049_xgb_shared001v2_features_optuna_search_min/feature_columns.csv
 - /kaggle/working/exp_20260523_049_xgb_shared001v2_features_optuna_search_min/shared001v2_features_049.csv
 - /kaggle/working/exp_20260523_049_xgb_shared001v2_features_optuna_search_min/fold_build_records.csv


In [15]:
# ============================================================
# Save memo_draft.yml
# ============================================================

print_section("Save memo_draft.yml")

if best_cv_auc > CFG.BASELINE_041_CV:
    decision = "success_go_to_050_refit"
elif best_cv_auc > CFG.BASELINE_016_CV:
    decision = "weak_hold_optional_050_once"
else:
    decision = "fail_do_not_deepen"

memo = f"""
experiment:
  id: {CFG.EXP_ID}
  competition: {CFG.COMPETITION}
  metric: {CFG.METRIC}
  created_at: {CFG.CREATED_AT}
  type: optuna_search
  status: completed

objective:
  summary: >
    041 XGB shared001v2 features構成を固定し、XGBoostの主要ハイパラだけをOptunaで探索する。
    目的はXGB branchの016/041を更新できる可能性を確認することであり、
    新規特徴量追加やstack変更は行わない。

base:
  experiment: exp_20260520_041_xgb_shared001v2_features_min
  model: XGBClassifier
  baseline_016:
    cv_auc: {CFG.BASELINE_016_CV:.16f}
    public_lb: {CFG.BASELINE_016_PUBLIC_LB:.5f}
  baseline_041:
    cv_auc: {CFG.BASELINE_041_CV:.16f}
    public_lb: {CFG.BASELINE_041_PUBLIC_LB:.5f}

fixed:
  feature_set: same_as_041
  folds: same_as_041
  original_rows: same_as_041
  target: {CFG.TARGET}
  metric: AUC
  forbidden:
    - Normalized_TyreLife
    - Normalized_TyreLife proxy
    - future endpoint proxy
    - pseudo label
    - new feature engineering
    - 039 LOG features

search_space:
  learning_rate: [0.01, 0.06, log]
  max_leaves: [32, 128]
  min_child_weight: [1.0, 30.0, log]
  subsample: [0.65, 0.95]
  colsample_bytree: [0.65, 0.95]
  reg_alpha: [0.0001, 10.0, log]
  reg_lambda: [1.0, 50.0, log]
  gamma: [0.0, 5.0]
  max_bin: [128, 256]

fixed_xgb_params:
  objective: binary:logistic
  eval_metric: auc
  n_estimators: {CFG.XGB_FIXED_PARAMS['n_estimators']}
  early_stopping_rounds: {CFG.XGB_FIXED_PARAMS['early_stopping_rounds']}
  grow_policy: lossguide
  max_depth: 0
  tree_method: {CFG.XGB_FIXED_PARAMS['tree_method']}
  device: {CFG.XGB_FIXED_PARAMS['device']}
  enable_categorical: true

optuna:
  direction: maximize
  n_trials_requested: {CFG.N_TRIALS}
  n_trials_completed: {len(study.trials)}
  timeout_sec: {CFG.TIMEOUT_SEC}
  sampler: TPESampler
  seed: {CFG.OPTUNA_SEED}
  model_seed_policy: single_seed_for_search
  optuna_model_seed: {CFG.OPTUNA_MODEL_SEED}

result:
  best_trial_number: {best_trial.number}
  best_cv_auc: {best_cv_auc:.16f}
  delta_vs_016_cv: {best_cv_auc - CFG.BASELINE_016_CV:+.16f}
  delta_vs_041_cv: {best_cv_auc - CFG.BASELINE_041_CV:+.16f}
  strong_if_cv_at_least: {CFG.STRONG_CV_THRESHOLD:.16f}
  decision: {decision}

best_params:
"""

for k, v in best_params.items():
    if isinstance(v, str):
        memo += f"  {k}: {v}\n"
    else:
        memo += f"  {k}: {v}\n"

memo += f"""

outputs:
  best_params: best_params_049.json
  trials: optuna_trials_049.csv
  study_summary: optuna_study_summary_049.json
  study_pkl: optuna_study_049.pkl
  feature_columns: feature_columns.csv
  shared001v2_features: shared001v2_features_049.csv
  memo_draft: memo_draft.yml

success_criteria:
  best_cv_must_exceed: {CFG.BASELINE_041_CV:.16f}
  strong_if_cv_at_least: {CFG.STRONG_CV_THRESHOLD:.16f}

next_step:
  if_success: >
    Create 050 refit notebook using best_params_049.json.
  if_fail: >
    Do not deepen XGB Optuna. Optionally run 050 once only if best params look reasonable.

notes:
  - >
    049 is search-only. It does not create final OOF/pred/submission.
  - >
    050 should use the same feature set and fold design as 041 and apply best_params_049.json.
  - >
    CV-only improvement is not enough for final adoption. 050 single LB and add050 blend must be checked.
"""

with open(CFG.OUTDIR / "memo_draft.yml", "w") as f:
    f.write(memo.strip() + "\n")

print(memo)
print("Saved:", CFG.OUTDIR / "memo_draft.yml")



Save memo_draft.yml

experiment:
  id: exp_20260523_049_xgb_shared001v2_features_optuna_search_min
  competition: PS S6E5 Predicting F1 Pit Stops
  metric: AUC
  created_at: 2026-05-23
  type: optuna_search
  status: completed

objective:
  summary: >
    041 XGB shared001v2 features構成を固定し、XGBoostの主要ハイパラだけをOptunaで探索する。
    目的はXGB branchの016/041を更新できる可能性を確認することであり、
    新規特徴量追加やstack変更は行わない。

base:
  experiment: exp_20260520_041_xgb_shared001v2_features_min
  model: XGBClassifier
  baseline_016:
    cv_auc: 0.9519855866173828
    public_lb: 0.95164
  baseline_041:
    cv_auc: 0.9520124407043763
    public_lb: 0.95173

fixed:
  feature_set: same_as_041
  folds: same_as_041
  original_rows: same_as_041
  target: PitNextLap
  metric: AUC
  forbidden:
    - Normalized_TyreLife
    - Normalized_TyreLife proxy
    - future endpoint proxy
    - pseudo label
    - new feature engineering
    - 039 LOG features

search_space:
  learning_rate: [0.01, 0.06, log]
  max_leaves: [32, 128]
  min_child

In [16]:
# ============================================================
# Preview
# ============================================================

print_section("Best Params Payload")
with open(CFG.OUTDIR / "best_params_049.json", "r") as f:
    best_preview = json.load(f)
print(json.dumps(best_preview, indent=2)[:4000])

print_section("Top Trials")
display(trials_df.sort_values("value", ascending=False).head(20))

print_section("Feature Columns Preview")
display(feature_df.head(30))

print_section("shared001v2 Features 049 Preview")
display(shared001v2_df.head(50))



Best Params Payload
{
  "experiment_id": "exp_20260523_049_xgb_shared001v2_features_optuna_search_min",
  "base_experiment": "exp_20260520_041_xgb_shared001v2_features_min",
  "created_at": "2026-05-23",
  "best_trial_number": 11,
  "best_cv_auc": 0.9522274536894715,
  "baseline_016_cv_auc": 0.9519855866173828,
  "baseline_041_cv_auc": 0.9520124407043763,
  "delta_vs_041_cv": 0.00021501298509518652,
  "strong_cv_threshold": 0.9522,
  "best_params": {
    "learning_rate": 0.010068327898866912,
    "max_leaves": 128,
    "min_child_weight": 7.2092861848805745,
    "subsample": 0.6582752198269224,
    "colsample_bytree": 0.6585993126442755,
    "reg_alpha": 0.0001235471477916111,
    "reg_lambda": 40.207068290141855,
    "gamma": 1.7896753305924635,
    "max_bin": 128
  },
  "effective_xgb_params_for_search_seed": {
    "n_estimators": 10000,
    "grow_policy": "lossguide",
    "max_depth": 0,
    "objective": "binary:logistic",
    "eval_metric": "auc",
    "tree_method": "hist",
    "d

,number,value,params_colsample_bytree,params_gamma,params_learning_rate,params_max_bin,params_max_leaves,params_min_child_weight,params_reg_alpha,params_reg_lambda,params_subsample,state,datetime_start,datetime_complete,duration,user_attrs_fold_aucs,user_attrs_fold_best_iters,user_attrs_mean_best_iter,user_attrs_std_auc
11,11,0.952227,0.658599,1.789675,0.010068,128,128,7.209286,0.000124,40.207068,0.658275,COMPLETE,2026-05-23 02:53:29.383383,2026-05-23 03:12:05.100115,0 days 00:18:35.716732,"[0.9534615032906542, 0.9510947980844374, 0.952...","[4392, 4283, 3469, 4214, 4635]",4198.6,0.000852
35,35,0.952221,0.668237,2.815019,0.010923,128,123,1.346891,0.018718,7.735375,0.667577,COMPLETE,2026-05-23 08:20:09.469752,2026-05-23 08:36:27.323949,0 days 00:16:17.854197,"[0.9534468874769253, 0.9511475666905062, 0.952...","[4699, 4153, 4048, 4683, 4730]",4462.6,0.000823
10,10,0.952219,0.654889,1.626800,0.010375,128,122,7.190751,0.000156,42.719062,0.655389,COMPLETE,2026-05-23 02:34:54.449910,2026-05-23 02:53:29.370898,0 days 00:18:34.920988,"[0.9534160316442416, 0.9511291034735547, 0.952...","[4838, 3918, 4161, 4075, 4522]",4302.8,0.000827
27,27,0.952215,0.696661,0.734944,0.011423,128,120,2.181620,0.003350,13.394911,0.680485,COMPLETE,2026-05-23 06:42:25.929108,2026-05-23 06:57:33.663494,0 days 00:15:07.734386,"[0.9535585001435681, 0.9511788114150688, 0.951...","[5051, 3742, 3225, 3438, 3982]",3887.6,0.000864
28,28,0.952185,0.698739,0.546845,0.011245,128,121,1.745944,0.002695,6.974141,0.675165,COMPLETE,2026-05-23 06:57:33.675090,2026-05-23 07:12:25.818618,0 days 00:14:52.143528,"[0.9535251856366739, 0.9510984602269187, 0.951...","[4455, 3827, 2898, 4123, 4512]",3963.0,0.000866
32,32,0.952182,0.704706,1.090063,0.011397,128,111,1.601143,0.000691,12.369631,0.670518,COMPLETE,2026-05-23 07:44:16.991578,2026-05-23 08:00:21.604466,0 days 00:16:04.612888,"[0.9534421246597974, 0.9510365181674413, 0.951...","[4378, 3931, 4090, 4383, 5163]",4389.0,0.000874
37,37,0.952180,0.725478,2.946876,0.010630,128,128,1.311340,0.015993,2.442471,0.688032,COMPLETE,2026-05-23 08:46:20.622420,2026-05-23 09:01:33.378797,0 days 00:15:12.756377,"[0.9534662819675723, 0.9512242085614356, 0.951...","[4920, 4795, 3247, 4311, 4210]",4296.6,0.000823
13,13,0.952162,0.651355,1.561175,0.010176,128,81,6.293997,0.000134,49.591613,0.708163,COMPLETE,2026-05-23 03:24:39.679047,2026-05-23 03:43:45.330159,0 days 00:19:05.651112,"[0.9534021730338506, 0.950996264694378, 0.9519...","[5853, 4549, 5250, 6236, 6342]",5646.0,0.000859
24,24,0.952159,0.678641,1.283166,0.011191,128,119,3.641202,0.001605,26.075313,0.670475,COMPLETE,2026-05-23 05:58:12.498957,2026-05-23 06:14:01.597309,0 days 00:15:49.098352,"[0.9533955294082173, 0.9510377979855852, 0.951...","[4011, 3878, 3430, 4081, 3796]",3839.2,0.000841
22,22,0.952158,0.675186,1.047695,0.011531,128,108,6.054787,0.000328,46.549155,0.714216,COMPLETE,2026-05-23 05:28:11.046968,2026-05-23 05:44:46.437085,0 days 00:16:35.390117,"[0.9534785684253835, 0.9510891582630127, 0.951...","[4617, 4053, 3491, 4327, 4174]",4132.4,0.000842



Feature Columns Preview


,feature,is_base,is_num_as_cat,is_digit,is_shared001v2_cat_041,is_shared001v2_count_041,is_frequency,is_target_encoding,contains_year,contains_race,contains_driver,contains_pitstop
0,Driver,True,False,False,False,False,False,False,False,False,True,False
1,Compound,True,False,False,False,False,False,False,False,False,False,False
2,Race,True,False,False,False,False,False,False,False,True,False,False
3,Year,True,False,False,False,False,False,False,True,False,False,False
4,PitStop,True,False,False,False,False,False,False,False,False,False,True
5,LapNumber,True,False,False,False,False,False,False,False,False,False,False
6,Stint,True,False,False,False,False,False,False,False,False,False,False
7,TyreLife,True,False,False,False,False,False,False,False,False,False,False
8,Position,True,False,False,False,False,False,False,False,False,False,False
9,LapTime (s),True,False,False,False,False,False,False,False,False,False,False



shared001v2 Features 049 Preview


,feature,kind
0,PitStop_cat_,cat_or_bin
1,RaceProgress_200_quantile_bin_,cat_or_bin
2,LapTime (s)_7_quantile_bin_,cat_or_bin
3,Race_Compound_,cat_or_bin
4,Race_Year_,cat_or_bin
5,LapNumber_cat_,cat_or_bin
6,Stint_cat_,cat_or_bin
7,TyreLife_cat_,cat_or_bin
8,Position_cat_,cat_or_bin
9,LapTime (s)_cat_,cat_or_bin
